In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sympy as sp
from sklearn.model_selection import train_test_split
import thermoift.PLOT_SETTINGS as ps
from pysr import PySRRegressor

In [ ]:
df = pd.read_csv("CSV_OLD/interfacial_results_cleaned.csv")


target      = ["P_bubble", "P_dew"]
exclude1    = ["gamma", "interfacial_thickness", "E_carbon dioxide", "E_argon", "E_hydrogen"]
exclude2    = ["x_carbon dioxide", "x_argon", "x_hydrogen", "y_carbon dioxide", "y_argon", "y_hydrogen"]
exclude3    = ["z_carbon dioxide"]
exclude4    = ["liquid_density", "vapor_density"]
exclude5    = ["pressure"]
exclude     = exclude1 + exclude2 + exclude3 +  exclude4 + exclude5 + target

features    = [col for col in df.columns if col not in target and col not in exclude]
print(features)

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=85)

print(f"Training: {X_train.shape[0]} rows")
print(f"Testing:  {X_test.shape[0]} rows")

In [ ]:
model_sr_bubble = PySRRegressor(
    niterations      = 500,
    binary_operators = [
        "+", "-", "*", "/",
        "safepow(x::T, y::T) where T = x >= 0 ? T(x^y) : T(NaN)",  # ← T() cast
    ],
    unary_operators  = [
        "square",
        "cube(x::T) where T = T(x^3)",
        "inv(x::T) where T = T(1/x)",
        "invsquare(x::T) where T = T(1/x^2)",
        "sqrt",
        "safecbrt(x::T) where T = x >= 0 ? T(x^(1/3)) : T(NaN)",   # ← T() cast
        "abs",
    ],
    extra_sympy_mappings = {
        "cube"      : lambda x: x**3,
        "inv"       : lambda x: 1/x,
        "invsquare" : lambda x: 1/x**2,
        "safecbrt"  : lambda x: x**(sp.Rational(1, 3)),
        "safepow"   : lambda x, y: x**y,
    },
    populations      = 40,
    maxsize          = 30,
    parsimony        = 0.001,
    random_state     = 42,
)

model_sr_dew = PySRRegressor(
    niterations      = 500,
    binary_operators = [
        "+", "-", "*", "/",
        "safepow(x::T, y::T) where T = x >= 0 ? T(x^y) : T(NaN)",
    ],
    unary_operators  = [
        "square",
        "cube(x::T) where T = T(x^3)",
        "inv(x::T) where T = T(1/x)",
        "invsquare(x::T) where T = T(1/x^2)",
        "sqrt",
        "safecbrt(x::T) where T = x >= 0 ? T(x^(1/3)) : T(NaN)",
        "abs",
    ],
    extra_sympy_mappings = {
        "cube"      : lambda x: x**3,
        "inv"       : lambda x: 1/x,
        "invsquare" : lambda x: 1/x**2,
        "safecbrt"  : lambda x: x**(sp.Rational(1, 3)),
        "safepow"   : lambda x, y: x**y,
    },
    populations      = 40,
    maxsize          = 30,
    parsimony        = 0.001,
    random_state     = 42,
)

# --- fit ---
model_sr_bubble.fit(X_train, y_train.iloc[:, 0])
model_sr_dew.fit(X_train,    y_train.iloc[:, 1])